In [31]:
# ============================================================================
# MANTENER SESIÓN ACTIVA (Opcional pero recomendado)
# ============================================================================

from IPython.display import Javascript

display(Javascript('''
    function ClickConnect(){
        console.log("Manteniendo conexión activa...");
        document.querySelector("#top-toolbar > colab-connect-button")?.shadowRoot.querySelector("#connect")?.click();
    }
    setInterval(ClickConnect, 60000);
'''))

print("✅ Auto-click activado para mantener sesión")

<IPython.core.display.Javascript object>

✅ Auto-click activado para mantener sesión


# Imports

In [ ]:
!pip install torch_geometric

In [2]:
import os
import numpy as np
import pandas as pd
import copy
import pprint
import gc
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader

import joblib
from datetime import datetime, timezone, timedelta

from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score, average_precision_score, roc_auc_score, confusion_matrix, precision_recall_curve
from scipy.special import expit # It is the optimized sigmoid of Scipy


In [3]:
# Change directory
os.chdir('/content/drive/MyDrive/nids-mitre/')

!pwd


/content/drive/MyDrive/nids-mitre


# Experiment Manager

In [4]:
class ExperimentManager:
    def __init__(self, log_file="./logs/experiments_log_gnn.csv", model_dir="./saved_models"):
        self.log_file = log_file
        self.model_dir = model_dir
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        os.makedirs(model_dir, exist_ok=True)

    def log_experiment(self, model_config=None, model_name=None, params=None, metrics=None, model_object=None):
        """
        Record an experiment in CSV format and optionally save the model.

        model_config: Recommended dict:
        - model_name (str)
        - type (str)
        - model_params (dict) # ONLY model hyperparameters
        - prob_threshold (float) # Decision threshold for probabilities (for metrics computation)
        - data_params (dict) # Optional
        - extra_params (dict) # Optional

        model_name: Name of the model (str)

        params: (legacy) Extra dict (for compatibility)
        metrics: Dict with results
        model_object: Model object (for saving)
        """

        if metrics is None:
            metrics = {}
        if params is None:
            params = {}

        # Timezone Argentina
        tz = timezone(timedelta(hours=-3))
        now = datetime.now(tz)

        # Input
        if model_config is not None:
            mname = model_config.get("model_name", model_name)
            mtype = model_config.get("type", None)
            model_params = model_config.get("model_params", {})
            threshold = model_config.get("prob_threshold", None)
            data_params = model_config.get("data_params", {})
            extra_params = model_config.get("extra_params", {})
        else:
            # legacy mode
            mname = model_name
            mtype = params.get("type", None)
            threshold = params.get("prob_threshold", None)
            model_params = params
            data_params = {}
            extra_params = {}

        # Create record
        entry = {
            "timestamp": now.strftime("%Y-%m-%d %H:%M:%S"),
            "model_name": mname,
        }
        if mtype is not None:
            entry["type"] = mtype
        if threshold is not None:
            entry["prob_threshold"] = threshold

        # Prefijos para evitar colisiones de nombres
        entry.update({f"hp_{k}": v for k, v in (model_params or {}).items()})
        entry.update({f"data_{k}": v for k, v in (data_params or {}).items()})
        entry.update({f"extra_{k}": v for k, v in {**extra_params, **params}.items()
                      if k not in ("type", "prob_threshold")})

        entry.update(metrics)

        # Save in csv
        df_new = pd.DataFrame([entry])
        if os.path.exists(self.log_file):
            df_new.to_csv(self.log_file, mode="a", header=False, index=False)
        else:
            df_new.to_csv(self.log_file, mode="w", header=True, index=False)

        print(f"Experiment recorded in {self.log_file}")

        # Save model
        if model_object is not None:
            metric_key = "AUC-PR" if "AUC-PR" in metrics else ("F1" if "F1" in metrics else None)
            metric_val = metrics.get(metric_key, 0) if metric_key else 0
            safe_key = metric_key or "metric"
            filename = f"{mname}_{now.strftime('%Y%m%d_%H%M')}_{safe_key}_{float(metric_val):.4f}"

            if "sklearn" in str(type(model_object)):
                import joblib
                joblib.dump(model_object, os.path.join(self.model_dir, f"{filename}.joblib"))
            else:
                import torch
                torch.save(model_object.state_dict(), os.path.join(self.model_dir, f"{filename}.pth"))

            print(f"Saved model: {filename}")


# Global instance
exp_manager = ExperimentManager()

# Load data

In [5]:
class NF_IDS_Dataset(Dataset):
    def __init__(self, root_dir, split='train'):
        # root_dir: (eg: "./dataset_processed")
        # split: 'train', 'val' or 'test'
        self.split_dir = os.path.join(root_dir, split)
        super().__init__(self.split_dir)

        # List files ordered numerically to respect the time
        self.files = sorted(
            [f for f in os.listdir(self.split_dir) if f.endswith('.pt')],
            key=lambda x: int(x.split('_')[1].split('.')[0])
        )

    def len(self):
        return len(self.files)

    def get(self, idx):
        data = torch.load(
            os.path.join(self.split_dir, self.files[idx]),
            weights_only=False # to allow loading complex graph objects
        )
        return data

# Models

## Static GNN

In [6]:
class StaticGNN(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, out_dim, dropout):
        super(StaticGNN, self).__init__()

        # LAYER 1: Input -> Hidden
        # Takes info of immediate neighbors
        self.gnn1 = GATv2Conv(
            in_channels=node_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=2,        # Use 2 attention heads for added robustness
            concat=False    # Averaged the printheads to maintain a 64 dimension
        )

        # LAYER 2: Hidden -> Hidden
        # Takes info about neighbors of neighbors (Context of 2 jumps)
        self.gnn2 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=1,
            concat=False
        )

        # EDGE CLASSIFIER
        # Input: U-Node (64) + V-Node (64) + Edge Features (32)
        self.classifier = nn.Sequential(
            nn.Linear(2 * hidden_dim + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout), # Dropout to avoid overfitting
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, out_dim)
        )


    def forward(self, x, edge_index, edge_attr):
        # Check empty graph
        if x.size(0) == 0: return torch.empty((0, 1), device=x.device)

        # 1. Fist gnn layer
        h1 = self.gnn1(x, edge_index, edge_attr=edge_attr)
        h1 = F.elu(h1)

        # 2. Second gnn layer
        # Note: passed h1 as node features, but used the SAME graph (edge_index)
        h2 = self.gnn2(h1, edge_index, edge_attr=edge_attr)
        h2 = F.elu(h2)

        # 3. Edge Classification
        # Take the final embeddings (h2) of the source and destination nodes
        src, dst = edge_index

        # Concatenate: [Context_SrcIP, Context_DstIP, Edge_Attributes]
        edge_rep = torch.cat([h2[src], h2[dst], edge_attr], dim=1)

        # Output
        return self.classifier(edge_rep)

## Static GNN bias init

In [10]:
class StaticGNN_biasinit(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, dropout, output_bias_init=None):
        super(StaticGNN_biasinit, self).__init__()

        # LAYER 1: Input -> Hidden
        # Takes info of immediate neighbors
        self.gnn1 = GATv2Conv(
            in_channels=node_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=2,        # Use 2 attention heads for added robustness
            concat=False    # Averaged the printheads to maintain a 64 dimension
        )

        # LAYER 2: Hidden -> Hidden
        # Takes info about neighbors of neighbors (Context of 2 jumps)
        self.gnn2 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=1,
            concat=False
        )

        # EDGE CLASSIFIER
        # Input: U-Node (64) + V-Node (64) + Edge Features (32)
        self.classifier = nn.Sequential(
            nn.Linear(2 * hidden_dim + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout), # Dropout to avoid overfitting
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

        # --- BIAS INITIALIZATION ---
        if output_bias_init is not None:
            # Access the last linear layer (index -1) and modify its bias
            self.classifier[-1].bias.data.fill_(output_bias_init)


    def forward(self, x, edge_index, edge_attr):
        # Check empty graph
        if x.size(0) == 0: return torch.empty((0, 1), device=x.device)

        # 1. Fist gnn layer
        h1 = self.gnn1(x, edge_index, edge_attr=edge_attr)
        h1 = F.elu(h1)

        # 2. Second gnn layer
        # Note: passed h1 as node features, but used the SAME graph (edge_index)
        h2 = self.gnn2(h1, edge_index, edge_attr=edge_attr)
        h2 = F.elu(h2)

        # 3. Edge Classification
        # Take the final embeddings (h2) of the source and destination nodes
        src, dst = edge_index

        # Concatenate: [Context_SrcIP, Context_DstIP, Edge_Attributes]
        edge_rep = torch.cat([h2[src], h2[dst], edge_attr], dim=1)

        # Output
        return self.classifier(edge_rep)

## ST-GNN

In [ ]:
class ST_GNN(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, out_dim, dropout):
        super(ST_GNN, self).__init__()

        self.hidden_dim = hidden_dim

        # LAYER 1 (Spatial): Input -> Hidden
        self.gnn1 = GATv2Conv(
            in_channels=node_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=2,
            concat=False # Averaged the printheads to maintain a 64 dimension
        )

        # LAYER 2 (Spatial): Hidden -> Hidden
        self.gnn2 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=1,
            concat=False
        )

        # TEMPORAL LAYER: GRU
        self.gru = nn.GRUCell(input_size=hidden_dim, hidden_size=hidden_dim)

        # EDGE CLASSIFIER
        # Input: story_U (64) + story_V (64) + edge_dim (32)
        self.classifier = nn.Sequential(
            nn.Linear(2 * hidden_dim + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, out_dim)
        )

        # Memory: Dictionary {global_id: hidden_state_tensor}
        self.node_memory = {}

    def get_memory(self, ids, device):
        """Restore previous states or return zeros if it is a new node"""
        mem_list = []
        for i in ids:
            i = i.item()
            if i in self.node_memory:
                mem_list.append(self.node_memory[i])
            else:
                mem_list.append(torch.zeros(self.hidden_dim, device=device))
        return torch.stack(mem_list)

    def update_memory(self, ids, h_new):
        """Save the new states in the global dictionary"""
        h_detached = h_new.detach() # Detach for Truncated BPTT
        for idx, global_id in enumerate(ids):
            self.node_memory[global_id.item()] = h_detached[idx]

    def detach_all_memory(self):
        """Call every 10 steps ("sequential batch") to clean old computational graphs"""
        for k in self.node_memory:
            self.node_memory[k] = self.node_memory[k].detach()

    def forward(self, x, edge_index, edge_attr, global_ids):
        # 0. Check empty graphs
        if x.size(0) == 0: return torch.empty((0, 1), device=x.device)

        # 1. Recover Memory (H_{t-1})
        h_prev = self.get_memory(global_ids, x.device)

        # 2. Spatial (GNNs) -> Z_t
        # Layer 1
        z = self.gnn1(x, edge_index, edge_attr=edge_attr)
        z = F.elu(z)
        # Layer 2
        z = self.gnn2(z, edge_index, edge_attr=edge_attr)
        z = F.elu(z)

        # 3. Temporal (GRU) -> H_t
        h_current = self.gru(z, h_prev)

        # 4. Update Memory
        self.update_memory(global_ids, h_current)

        # 5. Edge Classification (using the temporarily enriched hidden state)
        src, dst = edge_index
        edge_rep = torch.cat([h_current[src], h_current[dst], edge_attr], dim=1)

        return self.classifier(edge_rep)


## ST-GNN bias init

In [27]:
class ST_GNN_biasinit(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, dropout, output_bias_init=None):
        super(ST_GNN_biasinit, self).__init__()

        self.hidden_dim = hidden_dim

        # LAYER 1 (Spatial): Input -> Hidden
        self.gnn1 = GATv2Conv(
            in_channels=node_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=2,
            concat=False # Averaged the printheads to maintain a 64 dimension
        )

        # LAYER 2 (Spatial): Hidden -> Hidden
        self.gnn2 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=1,
            concat=False
        )

        # TEMPORAL LAYER: GRU
        self.gru = nn.GRUCell(input_size=hidden_dim, hidden_size=hidden_dim)

        # EDGE CLASSIFIER
        # Input: story_U (64) + story_V (64) + edge_dim (32)
        self.classifier = nn.Sequential(
            nn.Linear(2 * hidden_dim + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

        # --- BIAS INITIALIZATION ---
        if output_bias_init is not None:
            # Access the last linear layer (index -1) and modify its bias
            self.classifier[-1].bias.data.fill_(output_bias_init)

        # Memory: Dictionary {global_id: hidden_state_tensor}
        self.node_memory = {}

    def get_memory(self, ids, device):
        """Restore previous states or return zeros if it is a new node"""
        mem_list = []
        for i in ids:
            i = i.item()
            if i in self.node_memory:
                mem_list.append(self.node_memory[i])
            else:
                mem_list.append(torch.zeros(self.hidden_dim, device=device))
        return torch.stack(mem_list)

    def update_memory(self, ids, h_new):
        """Save the new states in the global dictionary"""
        h_detached = h_new.detach() # Detach for Truncated BPTT
        for idx, global_id in enumerate(ids):
            self.node_memory[global_id.item()] = h_detached[idx]

    def detach_all_memory(self):
        """Call every 10 steps ("sequential batch") to clean old computational graphs"""
        for k in self.node_memory:
            self.node_memory[k] = self.node_memory[k].detach()

    def forward(self, x, edge_index, edge_attr, global_ids):
        # 0. Check empty graphs
        if x.size(0) == 0: return torch.empty((0, 1), device=x.device)

        # 1. Recover Memory (H_{t-1})
        h_prev = self.get_memory(global_ids, x.device)

        # 2. Spatial (GNNs) -> Z_t
        # Layer 1
        z = self.gnn1(x, edge_index, edge_attr=edge_attr)
        z = F.elu(z)
        # Layer 2
        z = self.gnn2(z, edge_index, edge_attr=edge_attr)
        z = F.elu(z)

        # 3. Temporal (GRU) -> H_t
        h_current = self.gru(z, h_prev)

        # 4. Update Memory
        self.update_memory(global_ids, h_current)

        # 5. Edge Classification (using the temporarily enriched hidden state)
        src, dst = edge_index
        edge_rep = torch.cat([h_current[src], h_current[dst], edge_attr], dim=1)

        return self.classifier(edge_rep)


# Functions

This handles the complex logic: Truncated Backpropagation for ST-GNN and detailed metric calculations (including False Positives).

## Metrics

In [6]:
def calculate_metrics_gnn(y_true, y_pred_logits, prob_threshold=0.5):
    """
    y_true: List or array of actual labels (0 or 1).
    y_pred_logits: List or array of raw model outputs (before sigmoid).
    """
    y_true = np.array(y_true)
    logits = np.array(y_pred_logits)

    # Convert logits to probabilities, and to classes (0 or 1) depend on prob_threshold
    probs = expit(logits)
    preds = (probs > prob_threshold).astype(int)

    # Threshold-dependent metrics
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    f2 = fbeta_score(y_true, preds, beta=2, zero_division=0)

    # Threshold-independent metrics
    try:
        ap = average_precision_score(y_true, probs)
        roc = roc_auc_score(y_true, probs)
    except ValueError:
        # This occurs if y_true has only one class (e.g., only benign in the batch)
        ap = 0.0
        roc = 0.5

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {
        "Precision": prec, "Recall": rec, "F1": f1, "F2": f2, "AUC-PR": ap, "AUC-ROC": roc,
        "FPR": fpr, "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn), "Total_Flows": len(y_true)
    }



## Train and eval

In [7]:
def train_epoch(model, loader, optimizer, criterion, device, is_temporal=False, batch_steps=10):
    model.train()
    total_loss = 0
    steps = 0  # Count of valid graphs processed (not empty)

    # Loss accumulator for Truncated Backprop
    batch_loss = 0

    # Iterate over the Loader
    # batch_idx helps to know if we are on the last element
    for batch_idx, data in enumerate(loader):
        data = data.to(device)

        # If it is an empty graph, skip
        if data.x.shape[0] == 0:
            continue

        # Forward
        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        # Loss calculation
        loss = criterion(out.view(-1), data.y)
        batch_loss += loss
        steps += 1

        # Backpropagation:
        # 1. Each 'batch_steps' valid steps (e.g., every 10 graphs)
        # 2. Or if it is in the LAST batch of the loader (so as not to lose the remainder)
        is_last_batch = (batch_idx == len(loader) - 1)

        if (steps > 0 and steps % batch_steps == 0) or is_last_batch:
            # Only update if there's something accumulated
            if batch_loss > 0:
                optimizer.zero_grad()
                batch_loss.backward()
                optimizer.step()

                # Reset
                total_loss += batch_loss.item()
                batch_loss = 0

                # IMPORTANT for ST-GNN: Detach memory
                if is_temporal:
                    model.detach_all_memory()

    # Average loss per valid step
    return total_loss / steps if steps > 0 else 0

@torch.no_grad()
def evaluate(model, loader, criterion, device, prob_threshold, is_temporal=False):
    model.eval()
    all_logits = []
    all_trues = []
    total_loss = 0
    steps = 0

    # Clear the memory before starting the evaluation (only if it's temporal)
    if is_temporal:
        model.node_memory = {}

    # Don't need enumerate here because we're not doing backprop
    for data in loader:
        data = data.to(device)

        if data.x.shape[0] == 0: continue

        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        loss = criterion(out.view(-1), data.y)
        total_loss += loss.item()

        all_logits.extend(out.view(-1).cpu().numpy())
        all_trues.extend(data.y.cpu().numpy())
        steps += 1

    metrics = calculate_metrics_gnn(all_trues, all_logits, prob_threshold)
    metrics["Loss"] = total_loss / steps if steps > 0 else 0
    return metrics

## pos_weight

In [43]:
def check_class_balance(loader, name="Loader"):
    total_benign = 0
    total_attack = 0

    print(f"--- Analyzing {name} ---")
    for data in loader:
        # data.y is [1] or [N]
        y = data.y.view(-1)
        n_ones = y.sum().item()
        n_zeros = y.shape[0] - n_ones

        total_attack += n_ones
        total_benign += n_zeros

    print(f"Benign: {int(total_benign)}")
    print(f"Attack : {int(total_attack)}")

    if total_attack > 0:
        # POS_WEIGHT: (Negative / Positive)
        # Used to penalize errors in the minority class more severely
        ratio = total_benign / total_attack
        pos_weight_tensor = torch.tensor([ratio])

        # BIAS_INIT: log(Positives / Negatives)
        # Used to allow the network to "guess" the base probability from the start
        # Mathematically equivalent to: -math.log(ratio)
        bias_val = math.log(total_attack / total_benign)

        print(f"Ratio (pos_weight): {ratio:.2f}")
        print(f"Bias Init suggested: {bias_val:.4f}")

        return pos_weight_tensor, bias_val
    else:
        print("ALERT: There are no attacks on this dataset.")
        return None, None



## Optimal threshold

In [35]:
def find_optimal_threshold(model, loader, device, is_temporal=False):
    model.eval()

    if is_temporal:
        # Clear the memory before starting the evaluation (only if it's temporal)
        model.node_memory = {}

    all_probs = []
    all_trues = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)

            if is_temporal:
                out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
            else:
                out = model(data.x, data.edge_index, data.edge_attr)

            # Convert Logits to Probabilities (Sigmoid)
            probs = torch.sigmoid(out.view(-1))

            all_probs.extend(probs.cpu().numpy())
            all_trues.extend(data.y.cpu().numpy())

    all_trues = np.array(all_trues)
    all_probs = np.array(all_probs)

    # 1. Precision-Recall Curve
    precisions, recalls, thresholds = precision_recall_curve(all_trues, all_probs)

    # 2. Calculate F1 for each possible threshold
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    f1_scores = f1_scores[:-1]

    # 3. Find max
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]
    best_rec = recalls[best_idx]
    best_prec = precisions[best_idx]

    print(f"\nBEST THRESHOLD FOUND: {best_threshold:.4f}")
    print(f"   F1 Score : {best_f1:.4f}")
    print(f"   Recall   : {best_rec:.4f}")
    print(f"   Precision: {best_prec:.4f}")

    # Probability Diagnosis
    avg_benign = np.mean(all_probs[all_trues == 0])
    avg_attack = np.mean(all_probs[all_trues == 1])
    print(f"\nProbability Diagnosis:")
    print(f"   Average assigned to Benignos: {avg_benign:.4f}")
    print(f"   Average assigned to Attacks : {avg_attack:.4f}")

    return best_threshold

# Auxiliary

In [8]:
ROOT_PATH = "./dataset_processed"

In [9]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=False)

Train size: 1998 | Val size: 428


In [19]:
pos_weight_value, bias_value = check_class_balance(train_loader, "TRAIN")
print(f"pos_weight_value:{pos_weight_value}\n")
print(f"bias_value:{bias_value}\n")

check_class_balance(val_loader, "VAL")

--- Analyzing TRAIN ---
Benign: 992229
Attack : 49564
Ratio (pos_weight): 20.02
Bias Init suggested: -2.9967
pos_weight_value:tensor([20.0191])

bias_value:-2.9966891636638033

--- Analyzing VAL ---
Benign: 757206
Attack : 31329
Ratio (pos_weight): 24.17
Bias Init suggested: -3.1851


(tensor([24.1695]), -3.1850911570685216)

# Main

In [12]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 5
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.001

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 64
OUT_DIM = 1     # Binary output (logit): >0 is attack, <0 is benign
DROPOUT = 0.2

PROB_THRESHOLD = 0.5


print(f"Using device: {DEVICE}")


Using device: cpu


## Train static

### Bias init

In [18]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 30
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

PROB_THRESHOLD = 0.5




Using device: cpu


In [20]:
model_config = {
    "model_name": "StaticGNN_biasinit",
    "type": "GNN_traditional_BiasOn",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [21]:
exp_manager = ExperimentManager(log_file="./logs/experiments_log_gnn_biasinit.csv")

In [22]:
static_bias_model = StaticGNN_biasinit(**model_config['model_params']).to(DEVICE)

weight = torch.tensor([POS_WEIGHT]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=weight)

opt_static = torch.optim.Adam(static_bias_model.parameters(), lr=LR)


best_aucpr_static_bias = 0.0
best_wts_static_bias = copy.deepcopy(static_bias_model.state_dict())
best_metrics_static_bias = {}

print(f"--- Starting training: {model_config['model_params']} ---")

for epoch in range(EPOCHS):
    # Note: batch_steps here acts as a simple "gradient accumulation"
    loss = train_epoch(static_bias_model, train_loader, opt_static, criterion, DEVICE, is_temporal=False, batch_steps=BATCH_STEPS)
    metrics_static_bias = evaluate(static_bias_model, val_loader, criterion, DEVICE, prob_threshold=PROB_THRESHOLD, is_temporal=False)

    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Val AUC-PR: {metrics_static_bias['AUC-PR']:.4f} | Val F1: {metrics_static_bias['F1']:.4f} | Val Rec: {metrics_static_bias['Recall']:.4f}")

    if metrics_static_bias['AUC-PR'] > best_aucpr_static_bias:
        best_aucpr_static_bias = metrics_static_bias['AUC-PR']
        best_metrics_static_bias = metrics_static_bias
        best_wts_static_bias = copy.deepcopy(static_bias_model.state_dict())
        print(f"New record AUC-PR: {best_aucpr_static_bias:.4f} (FPR: {metrics_static_bias['FPR']:.4f})")


print(f"\nRestoring better weights (AUC-PR: {best_aucpr_static_bias:.4f})...")
static_bias_model.load_state_dict(best_wts_static_bias)
exp_manager.log_experiment(model_config=model_config, metrics=best_metrics_static_bias, model_object=static_bias_model)

print(f"\nBest AUC-PR for Static GNN biasinit obtained: {best_aucpr_static_bias:.4f}")

--- Starting training: {'node_dim': 16, 'edge_dim': 32, 'hidden_dim': 32, 'dropout': 0.2, 'output_bias_init': -2.9968} ---
Epoch 1 | Loss: 0.2219 | Val AUC-PR: 0.0498 | Val F1: 0.0000 | Val Rec: 0.0000
New record AUC-PR: 0.0498 (FPR: 0.0000)
Epoch 2 | Loss: 0.2010 | Val AUC-PR: 0.0422 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 3 | Loss: 0.1942 | Val AUC-PR: 0.0726 | Val F1: 0.0000 | Val Rec: 0.0000
New record AUC-PR: 0.0726 (FPR: 0.0000)
Epoch 4 | Loss: 0.2023 | Val AUC-PR: 0.0675 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 5 | Loss: 0.1933 | Val AUC-PR: 0.0562 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 6 | Loss: 0.1973 | Val AUC-PR: 0.0650 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 7 | Loss: 0.2107 | Val AUC-PR: 0.0549 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 8 | Loss: 0.2000 | Val AUC-PR: 0.0739 | Val F1: 0.0000 | Val Rec: 0.0000
New record AUC-PR: 0.0739 (FPR: 0.0000)
Epoch 9 | Loss: 0.2044 | Val AUC-PR: 0.0711 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 10 | Loss: 0.1956 | Val AUC-PR: 0.0704 |

In [24]:
optimal_thresh = find_optimal_threshold(static_bias_model, val_loader, DEVICE)


BEST THRESHOLD FOUND: 0.1659
   F1 Score : 0.2183
   Recall   : 0.1487
   Precision: 0.4098

Probability Diagnosis:
   Average assigned to Benignos: 0.0778
   Average assigned to Attacks : 0.1421


## Train ST-GNN

### Bias init

In [28]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 30
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

PROB_THRESHOLD = 0.5




Using device: cpu


In [29]:
model_config = {
    "model_name": "ST_GNN_biasinit",
    "type": "GNN_and_GRU_BiasOn",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    "prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [30]:
exp_manager = ExperimentManager(log_file="./logs/experiments_log_gnn_biasinit.csv")

In [33]:
st_bias_model = ST_GNN_biasinit(**model_config['model_params']).to(DEVICE)

weight = torch.tensor([POS_WEIGHT]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=weight)

opt_st = torch.optim.Adam(st_bias_model.parameters(), lr=LR)


best_aucpr_st_bias = 0.0
best_wts_st_bias = copy.deepcopy(st_bias_model.state_dict())
best_metrics_st_bias = {}

print(f"--- Starting training: {model_config['model_params']} ---")

for epoch in range(EPOCHS):
    # Note: batch_steps here acts as a simple "gradient accumulation"
    loss = train_epoch(st_bias_model, train_loader, opt_st, criterion, DEVICE, is_temporal=True, batch_steps=BATCH_STEPS)
    metrics_st_bias = evaluate(st_bias_model, val_loader, criterion, DEVICE, prob_threshold=PROB_THRESHOLD, is_temporal=True)

    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Val AUC-PR: {metrics_st_bias['AUC-PR']:.4f} | Val F1: {metrics_st_bias['F1']:.4f} | Val Rec: {metrics_st_bias['Recall']:.4f}")

    if metrics_st_bias['AUC-PR'] > best_aucpr_st_bias:
        best_aucpr_st_bias = metrics_st_bias['AUC-PR']
        best_metrics_st_bias = metrics_st_bias
        best_wts_st_bias = copy.deepcopy(st_bias_model.state_dict())
        print(f"New record AUC-PR: {best_aucpr_st_bias:.4f} (FPR: {metrics_st_bias['FPR']:.4f})")


print(f"\nRestoring better weights (AUC-PR: {best_aucpr_st_bias:.4f})...")
st_bias_model.load_state_dict(best_wts_st_bias)
exp_manager.log_experiment(model_config=model_config, metrics=best_metrics_st_bias, model_object=st_bias_model)

print(f"\nBest AUC-PR for ST GNN obtained: {best_aucpr_st_bias:.4f}")

--- Starting training: {'node_dim': 16, 'edge_dim': 32, 'hidden_dim': 32, 'dropout': 0.2, 'output_bias_init': -2.9968} ---
Epoch 1 | Loss: 0.2241 | Val AUC-PR: 0.0532 | Val F1: 0.0000 | Val Rec: 0.0000
New record AUC-PR: 0.0532 (FPR: 0.0000)
Epoch 2 | Loss: 0.2097 | Val AUC-PR: 0.0565 | Val F1: 0.0000 | Val Rec: 0.0000
New record AUC-PR: 0.0565 (FPR: 0.0000)
Epoch 3 | Loss: 0.2154 | Val AUC-PR: 0.0710 | Val F1: 0.0000 | Val Rec: 0.0000
New record AUC-PR: 0.0710 (FPR: 0.0000)
Epoch 4 | Loss: 0.2075 | Val AUC-PR: 0.0536 | Val F1: 0.0000 | Val Rec: 0.0000
Epoch 5 | Loss: 0.2002 | Val AUC-PR: 0.0713 | Val F1: 0.0000 | Val Rec: 0.0000
New record AUC-PR: 0.0713 (FPR: 0.0000)
Epoch 6 | Loss: 0.2022 | Val AUC-PR: 0.0717 | Val F1: 0.0000 | Val Rec: 0.0000
New record AUC-PR: 0.0717 (FPR: 0.0000)
Epoch 7 | Loss: 0.2127 | Val AUC-PR: 0.1849 | Val F1: 0.0000 | Val Rec: 0.0000
New record AUC-PR: 0.1849 (FPR: 0.0000)
Epoch 8 | Loss: 0.2004 | Val AUC-PR: 0.1902 | Val F1: 0.0000 | Val Rec: 0.0000
New r

In [36]:
optimal_thresh = find_optimal_threshold(st_bias_model, val_loader, DEVICE, True)


BEST THRESHOLD FOUND: 0.3805
   F1 Score : 0.2487
   Recall   : 0.1457
   Precision: 0.8502

Probability Diagnosis:
   Average assigned to Benignos: 0.0743
   Average assigned to Attacks : 0.1771
